In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

In [ ]:
events = pd.read_csv("economic_calendar_19_24.csv")

events.head()

In [ ]:
events_clean = events.copy()

events_clean["Date"] = pd.to_datetime(
    events_clean["Date"],
    format="%m/%d/%y",
    errors="coerce"
)

events_clean = events_clean.dropna(subset=["Date"])

events_clean.head()

In [ ]:
print(events_clean.shape)
print(events_clean["Date"].min(), "to", events_clean["Date"].max())

In [ ]:
def convert_economic_value(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    if value == "":
        return np.nan
    
    multiplier = 1
    
    if value.endswith("%"):
        value = value.replace("%", "")
        multiplier = 0.01
    
    elif value.endswith("K"):
        value = value.replace("K", "")
        multiplier = 1_000
    
    elif value.endswith("M"):
        value = value.replace("M", "")
        multiplier = 1_000_000
    
    elif value.endswith("B"):
        value = value.replace("B", "")
        multiplier = 1_000_000_000
    
    value = value.replace(",", "")
    
    try:
        return float(value) * multiplier
    except:
        return np.nan

In [ ]:
numeric_cols = ["Actual", "Previous", "Consensus", "Forecast"]

for col in numeric_cols:
    events_clean[col + "_num"] = events_clean[col].apply(convert_economic_value)

events_clean[
    ["Actual", "Actual_num", "Previous", "Previous_num", "Consensus", "Consensus_num", "Forecast", "Forecast_num"]
].head(20)

In [ ]:
def categorize_event(event):
    event = str(event).lower()
    
    if any(word in event for word in ["us-china", "china trade", "trade talks", "tariff", "geopolitical"]):
        return "Trade_War_Geopolitical"
    
    elif any(word in event for word in ["inflation", "cpi", "ppi", "consumer price", "producer price"]):
        return "Inflation"
    
    elif any(word in event for word in ["interest rate", "rate decision", "fomc", "fed"]):
        return "Interest_Rate"
    
    elif any(word in event for word in ["non farm", "payroll", "unemployment", "jobless", "employment", "claimant"]):
        return "Employment"
    
    elif any(word in event for word in ["gdp", "growth"]):
        return "Growth"
    
    elif any(word in event for word in ["pmi", "manufacturing", "industrial production", "factory"]):
        return "Manufacturing"
    
    elif any(word in event for word in ["retail", "consumer confidence", "business confidence"]):
        return "Consumer_Business_Confidence"
    
    elif any(word in event for word in ["trade", "exports", "imports", "balance"]):
        return "Trade"
    
    else:
        return "Other"

In [ ]:
def categorize_event(event):
    event = str(event).lower()
    
    if any(word in event for word in ["us-china", "china trade", "trade talks", "tariff", "geopolitical"]):
        return "Trade War/Geopolitical"
    
    elif any(word in event for word in ["inflation", "cpi", "ppi", "consumer price", "producer price"]):
        return "Inflation"
    
    elif any(word in event for word in ["interest rate", "rate decision", "fomc", "fed"]):
        return "Interest Rate"
    
    elif any(word in event for word in ["non farm", "payroll", "unemployment", "jobless", "employment", "claimant"]):
        return "Employment"
    
    elif any(word in event for word in ["gdp", "growth"]):
        return "Growth"
    
    elif any(word in event for word in ["pmi", "manufacturing", "industrial production", "factory"]):
        return "Manufacturing"
    
    elif any(word in event for word in ["retail", "consumer confidence", "business confidence"]):
        return "Consumer/Business Confidence"
    
    elif any(word in event for word in ["trade", "exports", "imports", "balance"]):
        return "Trade"
    
    else:
        return "Other"

In [ ]:
events_clean["Event_Category"] = events_clean["Event"].apply(categorize_event)

events_clean["Event_Category"].value_counts()

In [ ]:
def country_weight(country):
    country = str(country).upper()
    
    if country == "US":
        return 1.00
    elif country == "CN":
        return 0.80
    elif country in ["EA", "GB", "JP", "CA"]:
        return 0.60
    else:
        return 0.40

In [ ]:
def category_weight(category):
    if category == "Interest_Rate":
        return 1.00
    elif category == "Inflation":
        return 0.95
    elif category == "Employment":
        return 0.85
    elif category == "Trade_War_Geopolitical":
        return 0.80
    elif category == "Growth":
        return 0.70
    elif category == "Manufacturing":
        return 0.60
    elif category == "Trade":
        return 0.55
    elif category == "Consumer_Business_Confidence":
        return 0.50
    else:
        return 0.30

In [ ]:
events_clean["Country_Weight"] = events_clean["Country"].apply(country_weight)
events_clean["Category_Weight"] = events_clean["Event_Category"].apply(category_weight)

events_clean["Event_Importance"] = (
    events_clean["Country_Weight"] * events_clean["Category_Weight"]
)

events_clean[["Date", "Country", "Event", "Event_Category", "Event_Importance"]].head(20)

In [ ]:
def safe_normalized_difference(actual, reference):
    if pd.isna(actual) or pd.isna(reference):
        return np.nan
    
    if reference == 0:
        return np.nan
    
    return (actual - reference) / abs(reference)

In [ ]:
events_clean["Norm_Surprise_Consensus"] = events_clean.apply(
    lambda row: safe_normalized_difference(row["Actual_num"], row["Consensus_num"]),
    axis=1
)

events_clean["Norm_Surprise_Forecast"] = events_clean.apply(
    lambda row: safe_normalized_difference(row["Actual_num"], row["Forecast_num"]),
    axis=1
)

events_clean["Norm_Change_Previous"] = events_clean.apply(
    lambda row: safe_normalized_difference(row["Actual_num"], row["Previous_num"]),
    axis=1
)

In [ ]:
norm_cols = [
    "Norm_Surprise_Consensus",
    "Norm_Surprise_Forecast",
    "Norm_Change_Previous"
]

for col in norm_cols:
    events_clean[col] = events_clean[col].clip(-5, 5)

events_clean[norm_cols].describe()

In [ ]:
events_clean["Weighted_Norm_Surprise_Consensus"] = (
    events_clean["Norm_Surprise_Consensus"] * events_clean["Event_Importance"]
)

events_clean["Weighted_Norm_Surprise_Forecast"] = (
    events_clean["Norm_Surprise_Forecast"] * events_clean["Event_Importance"]
)

events_clean["Weighted_Norm_Change_Previous"] = (
    events_clean["Norm_Change_Previous"] * events_clean["Event_Importance"]
)

In [ ]:
daily_event_features = events_clean.groupby("Date").agg(
    event_count=("Event", "count"),
    avg_event_importance=("Event_Importance", "mean"),
    max_event_importance=("Event_Importance", "max"),
    
    avg_norm_surprise_consensus=("Norm_Surprise_Consensus", "mean"),
    avg_norm_surprise_forecast=("Norm_Surprise_Forecast", "mean"),
    avg_norm_change_previous=("Norm_Change_Previous", "mean"),
    
    weighted_norm_surprise_consensus=("Weighted_Norm_Surprise_Consensus", "mean"),
    weighted_norm_surprise_forecast=("Weighted_Norm_Surprise_Forecast", "mean"),
    weighted_norm_change_previous=("Weighted_Norm_Change_Previous", "mean"),
    
    us_event_count=("Country", lambda x: (x == "US").sum()),
    china_event_count=("Country", lambda x: (x == "CN").sum())
).reset_index()

daily_event_features = daily_event_features.fillna(0)

print(daily_event_features.shape)
daily_event_features.head()

In [ ]:
category_counts = pd.crosstab(
    events_clean["Date"],
    events_clean["Event_Category"]
).reset_index()

category_counts.head()

In [ ]:
daily_event_features = daily_event_features.merge(
    category_counts,
    on="Date",
    how="left"
)

daily_event_features = daily_event_features.fillna(0)

print(daily_event_features.shape)
daily_event_features.head()

In [ ]:
daily_event_features.columns

In [ ]:
daily_event_features.describe()

In [ ]:
gold_event = yf.download(
    "GC=F",
    start="2019-01-01",
    end="2023-12-31",
    interval="1d",
    auto_adjust=False,
    progress=False
)

if isinstance(gold_event.columns, pd.MultiIndex):
    gold_event.columns = gold_event.columns.get_level_values(0)

gold_event = gold_event[["Open", "High", "Low", "Close", "Volume"]]

print(gold_event.shape)
print(gold_event.index.min(), "to", gold_event.index.max())
gold_event.head()

In [ ]:
df_gold_event_model = gold_event.copy()

df_gold_event_model["Return"] = df_gold_event_model["Close"].pct_change()

df_gold_event_model["MA_7"] = df_gold_event_model["Close"].rolling(7).mean()
df_gold_event_model["MA_30"] = df_gold_event_model["Close"].rolling(30).mean()

df_gold_event_model["Volatility_7"] = df_gold_event_model["Return"].rolling(7).std()
df_gold_event_model["Volatility_30"] = df_gold_event_model["Return"].rolling(30).std()

df_gold_event_model["Momentum_7"] = df_gold_event_model["Close"] - df_gold_event_model["Close"].shift(7)
df_gold_event_model["Momentum_30"] = df_gold_event_model["Close"] - df_gold_event_model["Close"].shift(30)

df_gold_event_model["Volume_MA_7"] = df_gold_event_model["Volume"].rolling(7).mean()
df_gold_event_model["Volume_MA_30"] = df_gold_event_model["Volume"].rolling(30).mean()
df_gold_event_model["Volume_Change"] = df_gold_event_model["Volume"].pct_change()

df_gold_event_model = df_gold_event_model.replace([np.inf, -np.inf], np.nan)
df_gold_event_model = df_gold_event_model.dropna()

print(df_gold_event_model.shape)
df_gold_event_model.head()

In [ ]:
df_gold_event_model = df_gold_event_model.reset_index()
df_gold_event_model["Date"] = pd.to_datetime(df_gold_event_model["Date"])

daily_event_features["Date"] = pd.to_datetime(daily_event_features["Date"])

In [ ]:
gold_with_events = df_gold_event_model.merge(
    daily_event_features,
    on="Date",
    how="left"
)

# Fill missing event features with 0 for non-event days
event_feature_cols = daily_event_features.columns.drop("Date")

gold_with_events[event_feature_cols] = gold_with_events[event_feature_cols].fillna(0)

print(gold_with_events.shape)
gold_with_events.head()

In [ ]:
gold_with_events.isnull().sum()

In [ ]:
forecast_horizon = 7

gold_with_events["Future_Close"] = gold_with_events["Close"].shift(-forecast_horizon)

gold_with_events["Target_Return"] = (
    gold_with_events["Future_Close"] - gold_with_events["Close"]
) / gold_with_events["Close"]

gold_with_events = gold_with_events.dropna()

print(gold_with_events.shape)
gold_with_events[["Date", "Close", "Future_Close", "Target_Return"]].head()

In [ ]:
event_model_feature_columns = gold_with_events.columns.drop(
    ["Date", "Future_Close", "Target_Return"]
)

X_gold_events = gold_with_events[event_model_feature_columns]
y_gold_events = gold_with_events["Target_Return"]

print("Number of features:", len(event_model_feature_columns))
print("X_gold_events shape:", X_gold_events.shape)
print("y_gold_events shape:", y_gold_events.shape)

In [ ]:
train_size_events = int(len(gold_with_events) * 0.8)

X_train_events = X_gold_events.iloc[:train_size_events]
X_test_events = X_gold_events.iloc[train_size_events:]

y_train_events = y_gold_events.iloc[:train_size_events]
y_test_events = y_gold_events.iloc[train_size_events:]

print("X_train_events:", X_train_events.shape)
print("X_test_events:", X_test_events.shape)
print("y_train_events:", y_train_events.shape)
print("y_test_events:", y_test_events.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

gold_events_scaler = MinMaxScaler()

X_train_events_scaled = gold_events_scaler.fit_transform(X_train_events)
X_test_events_scaled = gold_events_scaler.transform(X_test_events)

print("X_train_events_scaled:", X_train_events_scaled.shape)
print("X_test_events_scaled:", X_test_events_scaled.shape)

print("Scaled train min:", X_train_events_scaled.min())
print("Scaled train max:", X_train_events_scaled.max())

In [ ]:
def create_sequences(X_data, y_data, time_steps=60):
    X_sequences = []
    y_sequences = []
    
    for i in range(time_steps, len(X_data)):
        X_sequences.append(X_data[i-time_steps:i])
        y_sequences.append(y_data[i])
    
    return np.array(X_sequences), np.array(y_sequences)

In [ ]:
time_steps = 60

X_train_events_seq, y_train_events_seq = create_sequences(
    X_train_events_scaled,
    y_train_events.values.reshape(-1, 1),
    time_steps
)

X_test_events_seq, y_test_events_seq = create_sequences(
    X_test_events_scaled,
    y_test_events.values.reshape(-1, 1),
    time_steps
)

print("X_train_events_seq:", X_train_events_seq.shape)
print("y_train_events_seq:", y_train_events_seq.shape)

print("X_test_events_seq:", X_test_events_seq.shape)
print("y_test_events_seq:", y_test_events_seq.shape)

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

In [ ]:
n_features_events = X_train_events_seq.shape[2]

gold_events_model = Sequential()

gold_events_model.add(LSTM(
    units=64,
    return_sequences=True,
    input_shape=(time_steps, n_features_events)
))

gold_events_model.add(Dropout(0.25))

gold_events_model.add(LSTM(
    units=32,
    return_sequences=False
))

gold_events_model.add(Dropout(0.25))

gold_events_model.add(Dense(16, activation="relu"))

gold_events_model.add(Dense(1))

In [ ]:
gold_events_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

gold_events_model.summary()

In [ ]:
gold_events_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
gold_events_history = gold_events_model.fit(
    X_train_events_seq,
    y_train_events_seq,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[gold_events_early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gold_events_history.history["loss"], label="Training Loss")
plt.plot(gold_events_history.history["val_loss"], label="Validation Loss")
plt.title("Gold + Economic Events LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("Final training loss:", gold_events_history.history["loss"][-1])
print("Final validation loss:", gold_events_history.history["val_loss"][-1])
print("Total epochs trained:", len(gold_events_history.history["loss"]))

In [ ]:
gold_events_predicted_return = gold_events_model.predict(X_test_events_seq)

print("Predicted return shape:", gold_events_predicted_return.shape)
print("Actual return shape:", y_test_events_seq.shape)

print("First 5 predicted returns:")
print(gold_events_predicted_return[:5])

print("\nFirst 5 actual returns:")
print(y_test_events_seq[:5])

In [ ]:
current_close_events_test = X_test_events["Close"].iloc[time_steps:].values.reshape(-1, 1)

actual_future_price_events = current_close_events_test * (1 + y_test_events_seq)
predicted_future_price_events = current_close_events_test * (1 + gold_events_predicted_return)

print("Current close shape:", current_close_events_test.shape)
print("Actual future price shape:", actual_future_price_events.shape)
print("Predicted future price shape:", predicted_future_price_events.shape)

In [ ]:
gold_events_lstm_mae = mean_absolute_error(
    actual_future_price_events,
    predicted_future_price_events
)

gold_events_lstm_rmse = np.sqrt(mean_squared_error(
    actual_future_price_events,
    predicted_future_price_events
))

gold_events_lstm_mape = np.mean(
    np.abs((actual_future_price_events - predicted_future_price_events) / actual_future_price_events)
) * 100

gold_events_lstm_r2 = r2_score(
    actual_future_price_events,
    predicted_future_price_events
)

print("Gold + Economic Events LSTM Metrics")
print("-----------------------------------")
print("MAE:", gold_events_lstm_mae)
print("RMSE:", gold_events_lstm_rmse)
print("MAPE:", gold_events_lstm_mape)
print("R² Score:", gold_events_lstm_r2)

In [ ]:
gold_events_baseline_pred = current_close_events_test.copy()

gold_events_baseline_mae = mean_absolute_error(
    actual_future_price_events,
    gold_events_baseline_pred
)

gold_events_baseline_rmse = np.sqrt(mean_squared_error(
    actual_future_price_events,
    gold_events_baseline_pred
))

gold_events_baseline_mape = np.mean(
    np.abs((actual_future_price_events - gold_events_baseline_pred) / actual_future_price_events)
) * 100

gold_events_baseline_r2 = r2_score(
    actual_future_price_events,
    gold_events_baseline_pred
)

print("Gold Events Naive Baseline Metrics")
print("----------------------------------")
print("MAE:", gold_events_baseline_mae)
print("RMSE:", gold_events_baseline_rmse)
print("MAPE:", gold_events_baseline_mape)
print("R² Score:", gold_events_baseline_r2)

In [ ]:
gold_events_comparison = pd.DataFrame({
    "Model": [
        "Gold Events Naive Baseline",
        "Gold + Economic Events LSTM"
    ],
    "MAE": [
        gold_events_baseline_mae,
        gold_events_lstm_mae
    ],
    "RMSE": [
        gold_events_baseline_rmse,
        gold_events_lstm_rmse
    ],
    "MAPE": [
        gold_events_baseline_mape,
        gold_events_lstm_mape
    ],
    "R2": [
        gold_events_baseline_r2,
        gold_events_lstm_r2
    ]
})

gold_events_comparison

### Economic Events Model Conclusion

Economic event features were engineered using event categories, country/category importance weights, and normalized surprise values from Actual, Consensus, Forecast, and Previous indicators.

However, the Gold + Economic Events LSTM underperformed the naive baseline on the event-period test set:

- Naive Baseline MAPE: 1.72%
- Gold + Economic Events LSTM MAPE: 2.18%

This indicates that adding macro-event features directly into the LSTM introduced noise and did not improve predictive performance. Therefore, the economic event features will be used as an explanatory risk layer in the dashboard, while the final price forecasting model remains the Gold Market-Only Return-Based LSTM.

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

gold_events_comparison.to_csv(
    "outputs/gold_events_model_comparison.csv",
    index=False
)

print("Saved event model comparison.")